<a href="https://colab.research.google.com/github/baokhanh546123/Build-Knowledge-Graph-Simple/blob/main/Test_KG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q pyspark==3.5.6 spark-nlp==5.5.0 graphframes-py==0.12.3 sentence-transformers  networkx matplotlib psutil datasets transformers

In [2]:
# -*- coding: utf-8 -*-
"""
Knowledge Graph pipeline (Wikipedia -> NER -> entity normalization -> co-occurrence graph)
RAM-bounded, staged version.  Target: peak RSS ~2-3 GB per stage.

Stages (moi stage chay trong 1 process rieng => JVM / torch duoc giai phong hoan toan):
  sample  : lay mau Wikipedia  -> kg_output/docs.parquet
  nlp     : Spark NLP (NER)    -> kg_output/occurrences, entity_freq      (JVM nang, khong co torch)
  embed   : chuan hoa thuc the -> kg_output/entity_mapping.parquet         (torch, KHONG co JVM)
  graph   : build vertices/edges bang Spark -> vertices, edges, edge_evidence (JVM nhe, khong co NLP)
  analyze : validate + degree + PageRank + export Gephi                    (GraphFrames)

Chay:
  python kg_pipeline.py --stage all --n-docs 1000
  python kg_pipeline.py --stage graph            # chay lai rieng 1 stage tu checkpoint parquet
  SPARK_MASTER=spark://spark-master:7077 python kg_pipeline.py --stage nlp   # tren Docker cluster
"""
from __future__ import annotations

import argparse
import contextlib
import gc
import json
import multiprocessing as mp
import os
import sys
import threading
import time
import traceback
from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
import psutil

SPARKNLP_PKG = "com.johnsnowlabs.nlp:spark-nlp_2.12:5.5.0"
GRAPHFRAMES_PKG = "io.graphframes:graphframes-spark3_2.12:0.12.3"
STAGES = ["sample", "nlp", "embed", "graph", "analyze"]


# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
@dataclass(frozen=True)
class Config:
    out_dir: str = "kg_output"

    # --- sampling
    n_docs: int = 100
    seed: int = 42
    shuffle_buffer: int = 10_000
    max_chars: int = 30_000          # cat bai qua dai -> chan 1 row "khong lo" lam phinh heap

    # --- Spark
    master: str = os.environ.get("SPARK_MASTER", "local[2]")
    nlp_driver_mem: str = "1536m"    # local mode: heap nay chua ca "executor"
    graph_driver_mem: str = "1g"
    executor_mem: str = "2g"         # chi dung khi master = spark://...
    partitions: int = 8              # nhieu partition nho => moi task giu it row trong RAM
    ner_batch: int = 16
    sentence_split_chars: int = 600  # cau dai hon se bi cat (tranh batch NER bi pad qua dai)
    min_name_len: int = 2

    # --- entity normalization (embedding)
    embed_model: str = "BAAI/bge-small-en-v1.5"
    sim_threshold: float = 0.84      # bge-small thuong "cao" -> nen thu 0.88-0.92 va xem merged_entities.csv
    embed_batch: int = 64
    embed_max_seq_len: int = 32      # ten thuc the rat ngan, 32 token la du
    sim_block_mb: int = 64           # tran RAM cho 1 khoi ma tran tuong dong
    torch_threads: int = 2

    # --- graph
    max_entities_per_sentence: int = 30   # 0 = khong gioi han; cau co >30 thuc the thuong la list/table rac
    use_mapping: bool = True              # False (--no-embed) => bo qua entity_mapping.parquet cu neu con ton tai


def P(cfg: Config, name: str) -> str:
    return os.path.join(cfg.out_dir, name)


# ----------------------------------------------------------------------------
# Resource monitor (RSS cua process + toan bo process con; GC cua JVM neu co Spark)
# ----------------------------------------------------------------------------
class ResourceMonitor:
    MB = 1024 * 1024

    def __init__(self, stage: str, out_dir: str, interval: float = 0.5):
        self.stage, self.out_dir, self.interval = stage, out_dir, interval
        self.me = psutil.Process(os.getpid())
        self.running = False
        self.peak = {"total": 0.0, "python": 0.0, "java": 0.0}
        self.start_mb = self.end_mb = 0.0
        self.gc0: dict = {}
        self.t0 = 0.0

    def _sample(self):
        total = py = jv = 0
        for p in [self.me] + self.me.children(recursive=True):
            try:
                rss, name = p.memory_info().rss, p.name().lower()
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
            total += rss
            if "java" in name:
                jv += rss
            else:
                py += rss
        return total / self.MB, py / self.MB, jv / self.MB

    def _loop(self):
        while self.running:
            t, p, j = self._sample()
            self.peak["total"] = max(self.peak["total"], t)
            self.peak["python"] = max(self.peak["python"], p)
            self.peak["java"] = max(self.peak["java"], j)
            time.sleep(self.interval)

    @staticmethod
    def _jvm_gc(spark):
        out = {}
        if spark is None:
            return out
        try:
            beans = spark.sparkContext._jvm.java.lang.management.ManagementFactory \
                .getGarbageCollectorMXBeans().iterator()
            while beans.hasNext():
                b = beans.next()
                out[str(b.getName())] = (max(int(b.getCollectionTime()), 0), max(int(b.getCollectionCount()), 0))
        except Exception as e:  # noqa: BLE001
            print(f"[monitor] cannot read GC: {e}")
        return out

    def start(self, spark=None):
        self.t0 = time.time()
        self.start_mb = self._sample()[0]
        self.peak["total"] = self.start_mb
        self.gc0 = self._jvm_gc(spark)
        self.running = True
        self.th = threading.Thread(target=self._loop, daemon=True)
        self.th.start()

    def stop(self, spark=None) -> dict:
        self.running = False
        self.th.join()
        self.end_mb = self._sample()[0]
        self.peak["total"] = max(self.peak["total"], self.end_mb)
        gc1 = self._jvm_gc(spark)
        gc_ms = sum(gc1[k][0] - self.gc0.get(k, (0, 0))[0] for k in gc1)
        gc_n = sum(gc1[k][1] - self.gc0.get(k, (0, 0))[1] for k in gc1)
        rep = {
            "stage": self.stage,
            "seconds": round(time.time() - self.t0, 1),
            "rss_start_mb": round(self.start_mb, 1),
            "rss_end_mb": round(self.end_mb, 1),
            "peak_rss_mb": round(self.peak["total"], 1),
            "peak_python_mb": round(self.peak["python"], 1),
            "peak_java_mb": round(self.peak["java"], 1),
            "jvm_gc_ms": gc_ms,
            "jvm_gc_count": gc_n,
        }
        print(f"\n===== RESOURCE [{self.stage}] =====")
        for k, v in rep.items():
            if k != "stage":
                print(f"{k:<16}: {v}")
        os.makedirs(self.out_dir, exist_ok=True)
        with open(os.path.join(self.out_dir, f"metrics_{self.stage}.json"), "w") as f:
            json.dump(rep, f, indent=2)
        return rep


# ----------------------------------------------------------------------------
# Spark helper
# ----------------------------------------------------------------------------
def build_spark(cfg: Config, app: str, driver_mem: str, packages=()):
    """Tao SparkSession voi heap bi chan.

    Luu y: spark.driver.memory PHAI di vao PYSPARK_SUBMIT_ARGS (truoc khi JVM khoi dong).
    spark.conf.set("spark.driver.memory", ...) sau khi session da chay se KHONG co tac dung,
    va sparknlp.start() mac dinh memory="16G".
    """
    args = [f"--driver-memory {driver_mem}"]
    if packages:
        args.append(f"--packages {','.join(packages)}")
    os.environ["PYSPARK_SUBMIT_ARGS"] = " ".join(args + ["pyspark-shell"])

    from pyspark.sql import SparkSession

    b = (
        SparkSession.builder.appName(app)
        .master(cfg.master)
        .config("spark.sql.shuffle.partitions", str(cfg.partitions))
        .config("spark.sql.autoBroadcastJoinThreshold", "-1")        # chi broadcast khi co hint ro rang
        .config("spark.sql.execution.arrow.pyspark.enabled", "false")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("spark.kryoserializer.buffer.max", "256m")           # spark-nlp mac dinh 2000m
        .config("spark.driver.maxResultSize", "512m")                # spark-nlp mac dinh 0 (khong gioi han)
        .config("spark.memory.fraction", "0.4")                      # chua cho object cua annotator (user memory)
        .config("spark.ui.enabled", "false")
        .config("spark.ui.showConsoleProgress", "false")   # bo thanh tien do [Stage x:>] lam ban log
    )
    if cfg.master.startswith("spark://"):
        b = b.config("spark.executor.memory", cfg.executor_mem).config("spark.executor.cores", "1")
    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    return spark


def acquire_spark(cfg: Config, spark, app: str, driver_mem: str, packages=()):
    """Dung lai SparkSession duoc truyen vao; neu spark=None thi tu tao.
    Tra ve (spark, owned): chi stage nao tu tao (owned=True) moi duoc stop() khi xong."""
    if spark is not None:
        return spark, False
    return build_spark(cfg, app, driver_mem, packages), True


def norm_name(col):
    from pyspark.sql import functions as F
    return F.lower(F.trim(F.regexp_replace(col, r"\s+", " ")))


# ----------------------------------------------------------------------------
# Stage 0: sample
# ----------------------------------------------------------------------------
def clip_text(text: str, limit: int) -> str:
    if not limit or len(text) <= limit:
        return text
    cut = text.rfind(". ", 0, limit)
    return text[: cut + 1] if cut > limit // 2 else text[:limit]


def stage_sample(cfg: Config):
    from datasets import load_dataset

    ds = load_dataset("wikimedia/wikipedia", data_dir="20231101.en", token=True, streaming=True)["train"]
    ds = ds.shuffle(seed=cfg.seed, buffer_size=cfg.shuffle_buffer).take(cfg.n_docs)

    rows = []
    for r in ds:
        text = (r.get("text") or "").strip()
        if not text:
            continue
        rows.append({
            "document_id": str(r.get("id", "")).strip(),
            "title": r.get("title") or "",
            "source_link": r.get("url") or "",
            "content": clip_text(text, cfg.max_chars),
        })
    os.makedirs(cfg.out_dir, exist_ok=True)
    df = pd.DataFrame(rows)
    df.to_parquet(P(cfg, "docs.parquet"), index=False)
    print(f"docs: {len(df)} | avg chars: {df.content.str.len().mean():.0f} | max chars: {df.content.str.len().max()}")


# ----------------------------------------------------------------------------
# Building blocks (dung chung boi stage_* va kg_mapreduce_walkthrough.py)
# ----------------------------------------------------------------------------
PAIR_EXPR = """
flatten(transform(sequence(0, size(ids) - 2), i ->
    transform(sequence(i + 1, size(ids) - 1), j -> named_struct('src', ids[i], 'dst', ids[j]))))
"""


def ner_pipeline(cfg: Config):
    """Pipeline Spark NLP: document -> sentence -> token -> GloVe -> NER -> ner_chunk."""
    from pyspark.ml import Pipeline
    from sparknlp.annotator import (NerConverter, NerDLModel, SentenceDetector, Tokenizer,
                                    WordEmbeddingsModel)
    from sparknlp.base import DocumentAssembler

    return Pipeline(stages=[
        DocumentAssembler().setInputCol("content").setOutputCol("document"),
        SentenceDetector().setInputCols(["document"]).setOutputCol("sentence")
        .setSplitLength(cfg.sentence_split_chars),
        Tokenizer().setInputCols(["sentence"]).setOutputCol("token"),
        WordEmbeddingsModel.pretrained("glove_100d", "en")
        .setInputCols(["sentence", "token"]).setOutputCol("embeddings"),
        NerDLModel.pretrained("ner_dl", "en")
        .setInputCols(["sentence", "token", "embeddings"]).setOutputCol("ner")
        .setBatchSize(cfg.ner_batch),
        NerConverter().setInputCols(["sentence", "token", "ner"]).setOutputCol("ner_chunk"),
    ])


def occurrences_from(annotated, cfg: Config):
    """annotated: DataFrame sau pipeline NER. Chi giu cot can cho do thi (flatMap: 1 doc -> n entity)."""
    from pyspark.sql import functions as F

    return (
        annotated
        .select("document_id", F.explode("ner_chunk").alias("c"))
        .select(
            "document_id",
            F.col("c.result").alias("surface"),
            F.col("c.metadata")["entity"].alias("entity_type"),
            F.col("c.metadata")["sentence"].cast("int").alias("sentence_id"),
            F.col("c.begin").alias("begin"),
            F.col("c.end").alias("end"),
        )
        .where(F.col("surface").isNotNull() & F.col("entity_type").isNotNull())
        .withColumn("name_norm", norm_name(F.col("surface")))
        .where(F.length("name_norm") >= cfg.min_name_len)
    )


def entity_freq_from(occ):
    """Reduce kieu word-count: key = (entity_type, name_norm), value = so lan xuat hien."""
    from pyspark.sql import functions as F
    return occ.groupBy("entity_type", "name_norm").agg(F.count("*").alias("freq"))


def with_entity_id(occ, mapping=None):
    """Map-side (broadcast) join voi bang canonical => cot `id` = <TYPE>::<canonical>. Khong shuffle."""
    from pyspark.sql import functions as F

    if mapping is not None:
        occ = (occ.join(F.broadcast(mapping), ["entity_type", "name_norm"], "left")
                  .withColumn("canonical", F.coalesce("canonical", "name_norm")))
    else:
        occ = occ.withColumn("canonical", F.col("name_norm"))
    return (occ.withColumn("id", F.concat_ws("::", "entity_type", "canonical"))
               .select("document_id", "sentence_id", "id", "entity_type", "surface"))


def vertices_from(occ_id):
    """2 reduce lien tiep: (id,type,surface)->count, roi (id,type)->sum + dang viet pho bien nhat."""
    from pyspark.sql import functions as F

    surf = occ_id.groupBy("id", "entity_type", "surface").agg(F.count("*").alias("c"))
    return (surf.groupBy("id", "entity_type")
            .agg(F.sum("c").alias("frequency"),
                 F.max(F.struct("c", "surface")).getField("surface").alias("name"))
            .select("id", "name", "entity_type", "frequency"))


def sentence_entities_from(occ_id, cfg: Config):
    """Reduce theo key = (document_id, sentence_id): gom cac entity id (da sort) cung 1 cau."""
    from pyspark.sql import functions as F

    per_sentence = (occ_id.groupBy("document_id", "sentence_id")
                    .agg(F.array_sort(F.collect_set("id")).alias("ids"))
                    .where(F.size("ids") >= 2))
    if cfg.max_entities_per_sentence:
        per_sentence = per_sentence.where(F.size("ids") <= cfg.max_entities_per_sentence)
    return per_sentence


def pairs_from(per_sentence):
    """flatMap: moi cau co k entity -> k(k-1)/2 cap (src < dst)."""
    from pyspark.sql import functions as F

    return (per_sentence
            .select("document_id", "sentence_id", F.explode(F.expr(PAIR_EXPR)).alias("p"))
            .select("document_id", "sentence_id", F.col("p.src").alias("src"), F.col("p.dst").alias("dst")))


def edges_from(pairs):
    """Reduce theo key = (src, dst): so cau va so tai lieu dong xuat hien."""
    from pyspark.sql import functions as F

    return (pairs.groupBy("src", "dst")
            .agg(F.count("*").alias("sentence_count"),
                 F.countDistinct("document_id").alias("doc_count"))
            .withColumn("relationship", F.lit("CO_OCCURS"))
            .withColumn("src_type", F.substring_index("src", "::", 1))
            .withColumn("dst_type", F.substring_index("dst", "::", 1)))


# ----------------------------------------------------------------------------
# Stage 1: NLP (Spark NLP) -> occurrences  (khong cache, khong giu annotation nao trong RAM)
# ----------------------------------------------------------------------------
def stage_nlp(cfg: Config, spark=None):
    mon = ResourceMonitor("nlp", cfg.out_dir)
    spark, owned = acquire_spark(cfg, spark, "kg-nlp", cfg.nlp_driver_mem, [SPARKNLP_PKG])
    mon.start(spark)
    try:
        docs = (
            spark.read.parquet(P(cfg, "docs.parquet"))
            .select("document_id", "content")
            .repartition(cfg.partitions)      # 1 file parquet = 1 partition => khong co repartition thi chi chay 1 task
        )
        model = ner_pipeline(cfg).fit(docs)   # toan bo la Transformer da train san => khong chay job nao
        # Cac cot document/sentence/token/embeddings/ner KHONG bao gio roi khoi task.
        occ = occurrences_from(model.transform(docs), cfg)
        occ.write.mode("overwrite").parquet(P(cfg, "occurrences"))

        # Bang tan suat (nho) cho stage embed: doc tu parquet, khong tinh lai NLP
        occ = spark.read.parquet(P(cfg, "occurrences"))
        entity_freq_from(occ).write.mode("overwrite").parquet(P(cfg, "entity_freq"))

        print(f"occurrences: {occ.count()} | unique (type,name): "
              f"{spark.read.parquet(P(cfg, 'entity_freq')).count()}")
    finally:
        mon.stop(spark)
        if owned:
            spark.stop()


# ----------------------------------------------------------------------------
# Stage 2: entity normalization (pure Python, khong JVM)
# ----------------------------------------------------------------------------
def greedy_leader_clustering(emb: np.ndarray, threshold: float, max_block_mb: int = 64) -> np.ndarray:
    """Giong het thuat toan cu (leader dau tien, gom cac j>i chua gan co sim >= threshold)
    nhung bo nho la O(block * n) thay vi O(n^2).

    emb : (n, d) float32, da L2-normalize, sap theo do uu tien giam dan (leader = thuc the pho bien nhat).
    return: leader[i] = chi so cua dai dien cum chua i.
    """
    n = emb.shape[0]
    leader = np.arange(n)
    assigned = np.zeros(n, dtype=bool)
    block = max(16, int(max_block_mb * 1024 * 1024 // (4 * max(n, 1))))
    for s in range(0, n, block):
        e = min(s + block, n)
        if assigned[s:e].all():                    # ca khoi da la thanh vien cum truoc => khong co leader moi
            continue
        sims = emb[s:e] @ emb[s:].T               # (e-s, n-s) float32
        for i in range(s, e):
            if assigned[i]:
                continue
            row = sims[i - s, i - s + 1:]
            hit = np.flatnonzero((row >= threshold) & ~assigned[i + 1:]) + i + 1
            if hit.size:
                leader[hit] = i
                assigned[hit] = True
            assigned[i] = True
        del sims
    return leader


def stage_embed(cfg: Config):
    mon = ResourceMonitor("embed", cfg.out_dir)
    mon.start()
    try:
        import torch
        from sentence_transformers import SentenceTransformer

        torch.set_num_threads(cfg.torch_threads)
        freq = pd.read_parquet(P(cfg, "entity_freq"))
        print(f"Unique (type, name): {len(freq)}")

        model = SentenceTransformer(cfg.embed_model, device="cpu")
        model.max_seq_length = cfg.embed_max_seq_len

        parts = []
        # Cum rieng theo entity_type: (a) dung nghia hon (LOC "Washington" khong bi doi thanh PER),
        # (b) ma tran tuong dong nho di ~k^2 lan.
        for etype, g in freq.groupby("entity_type"):
            g = g.sort_values(["freq", "name_norm"], ascending=[False, True])
            names = g["name_norm"].tolist()
            with torch.inference_mode():
                emb = model.encode(names, batch_size=cfg.embed_batch, normalize_embeddings=True,
                                   convert_to_numpy=True, show_progress_bar=False).astype(np.float32, copy=False)
            leader = greedy_leader_clustering(emb, cfg.sim_threshold, cfg.sim_block_mb)
            parts.append(pd.DataFrame({
                "entity_type": etype,
                "name_norm": names,
                "canonical": [names[i] for i in leader],
            }))
            print(f"  {etype}: {len(names)} names -> {len(set(leader.tolist()))} clusters")
            del emb, leader
            gc.collect()

        mapping = pd.concat(parts, ignore_index=True)
        mapping.to_parquet(P(cfg, "entity_mapping.parquet"), index=False)

        merged = mapping[mapping.name_norm != mapping.canonical]
        merged.to_csv(P(cfg, "merged_entities.csv"), index=False)     # de kiem tra bang mat, khong in ra man hinh
        print(f"merged names: {len(merged)} (xem merged_entities.csv)")
        print(merged.groupby(["entity_type", "canonical"]).size().sort_values(ascending=False).head(10))
    finally:
        mon.stop()


# ----------------------------------------------------------------------------
# Stage 3: build graph (Spark, khong NLP)
# ----------------------------------------------------------------------------
def stage_graph(cfg: Config, spark=None):
    mon = ResourceMonitor("graph", cfg.out_dir)
    spark, owned = acquire_spark(cfg, spark, "kg-graph", cfg.graph_driver_mem)
    mon.start(spark)
    try:
        occ = spark.read.parquet(P(cfg, "occurrences"))
        map_path = P(cfg, "entity_mapping.parquet")
        mapping = spark.read.parquet(map_path) if (cfg.use_mapping and os.path.exists(map_path)) else None
        occ_id = with_entity_id(occ, mapping)

        vertices_from(occ_id).write.mode("overwrite").parquet(P(cfg, "vertices"))

        pairs = pairs_from(sentence_entities_from(occ_id, cfg))
        pairs.write.mode("overwrite").parquet(P(cfg, "edge_evidence"))    # provenance: (src,dst,doc,sentence)
        edges_from(spark.read.parquet(P(cfg, "edge_evidence"))).write.mode("overwrite").parquet(P(cfg, "edges"))

        print("\n=== Final Stats ===")
        print("Vertices:", spark.read.parquet(P(cfg, "vertices")).count())
        print("Edges   :", spark.read.parquet(P(cfg, "edges")).count())
    finally:
        mon.stop(spark)
        if owned:
            spark.stop()


# ----------------------------------------------------------------------------
# Stage 4: analyze (GraphFrames)
# ----------------------------------------------------------------------------
def stage_analyze(cfg: Config, plot_top: int = 0, spark=None):
    mon = ResourceMonitor("analyze", cfg.out_dir)
    spark, owned = acquire_spark(cfg, spark, "kg-analyze", cfg.graph_driver_mem, [GRAPHFRAMES_PKG])
    mon.start(spark)
    try:
        from graphframes import GraphFrame
        from pyspark.sql import functions as F

        v = spark.read.parquet(P(cfg, "vertices"))
        e = spark.read.parquet(P(cfg, "edges"))

        # ---- validate
        report = {
            "null_vertices": v.where(F.col("id").isNull()).count(),
            "null_edges": e.where(F.col("src").isNull() | F.col("dst").isNull()).count(),
            "duplicate_vertices": v.groupBy("id").count().where("count > 1").count(),
            "dangling_edges": (e.select("src").union(e.select("dst")).distinct()
                               .join(v.select(F.col("id").alias("src")), "src", "left_anti").count()),
            "self_loops": e.where(F.col("src") == F.col("dst")).count(),
        }
        for k, val in report.items():
            print(f"{k:<20}: {val}")
        if any(report.values()):
            raise ValueError("Invalid Knowledge Graph")
        print("Graph validation: PASSED")

        # ---- degree: canh luu 1 chieu (src < dst) nen in+out = degree vo huong dung.
        # (Ban cu degree tren do thi da 2 chieu => bi nhan doi.)
        g = GraphFrame(v, e)
        degrees = g.degrees.join(v, "id").select("id", "name", "entity_type", "degree")
        top_deg = degrees.orderBy(F.desc("degree")).limit(20).toPandas()
        print(top_deg.to_string(index=False))

        # ---- PageRank can do thi vo huong -> doi xung hoa canh
        sym = e.select("src", "dst").unionByName(e.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
        pr = GraphFrame(v, sym).pageRank(resetProbability=0.15, maxIter=10)
        top_pr = pr.vertices.orderBy(F.desc("pagerank")).select("id", "name", "entity_type", "pagerank") \
                   .limit(20).toPandas()
        print(top_pr.to_string(index=False))

        n_v, n_e = v.count(), e.count()
        avg_deg = 2.0 * n_e / max(n_v, 1)
        print(f"Vertices: {n_v} | Edges: {n_e} | Avg degree: {avg_deg:.2f} | Max degree: {int(top_deg.degree.max())}")

        # ---- export Gephi (CSV import truc tiep, khong can part-*.csv va duong dan /content/)
        gephi = P(cfg, "gephi")
        os.makedirs(gephi, exist_ok=True)
        nodes_pd = v.select(F.col("id").alias("Id"), F.col("name").alias("Label"), "entity_type", "frequency").toPandas()
        edges_pd = e.select(F.col("src").alias("Source"), F.col("dst").alias("Target"),
                            F.col("sentence_count").alias("Weight")).toPandas()
        nodes_pd.to_csv(os.path.join(gephi, "nodes.csv"), index=False)
        edges_pd.to_csv(os.path.join(gephi, "edges.csv"), index=False)
        print(f"Gephi export -> {gephi}")

        if plot_top:
            plot_top_subgraph(nodes_pd, edges_pd, degrees.toPandas(), plot_top, os.path.join(cfg.out_dir, "graph_top.png"))
    finally:
        mon.stop(spark)
        if owned:
            spark.stop()


def plot_top_subgraph(nodes_pd, edges_pd, degrees_pd, top_n: int, path: str):
    """Ve top-N nut theo degree (ve 20k nut bang spring_layout vua cham vua khong doc duoc)."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import networkx as nx

    keep = set(degrees_pd.nlargest(top_n, "degree")["id"])
    sub = edges_pd[edges_pd.Source.isin(keep) & edges_pd.Target.isin(keep)]
    G = nx.from_pandas_edgelist(sub, "Source", "Target", edge_attr="Weight")
    pos = nx.spring_layout(G, k=0.4, iterations=50, seed=42)
    plt.figure(figsize=(14, 10))
    nx.draw_networkx_edges(G, pos, alpha=0.3, width=0.8)
    nx.draw_networkx_nodes(G, pos, node_size=120, node_color="skyblue")
    nx.draw_networkx_labels(G, pos, labels={n: n.split("::")[-1] for n in G}, font_size=7)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"plot -> {path}")


# ----------------------------------------------------------------------------
# Entry points: dung duoc ca trong notebook (Colab/Jupyter) lan CLI
# ----------------------------------------------------------------------------
def _in_notebook() -> bool:
    return "ipykernel" in sys.modules or "google.colab" in sys.modules


def _call_stage(stage: str, cfg: Config, plot: int = 0, spark=None) -> None:
    """`spark` (neu co) la session dung chung; stage sample/embed khong dung Spark nen bo qua."""
    if stage == "analyze":
        stage_analyze(cfg, plot_top=plot, spark=spark)
    elif stage == "nlp":
        stage_nlp(cfg, spark=spark)
    elif stage == "graph":
        stage_graph(cfg, spark=spark)
    else:
        {"sample": stage_sample, "embed": stage_embed}[stage](cfg)


def _stage_child(stage: str, cfg: Config, plot: int, log_path: str) -> None:
    """Chay trong process con (fork). Moi thu in ra duoc ghi vao file log de process cha doc lai."""
    with open(log_path, "w", buffering=1) as f, contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        try:
            _call_stage(stage, cfg, plot)
        except BaseException:
            traceback.print_exc()
            sys.exit(1)


def _drain(path: str, pos: int) -> int:
    """In phan log moi (tu byte `pos`) len man hinh / output cua cell, tra ve vi tri moi."""
    try:
        with open(path, "rb") as f:
            f.seek(pos)
            chunk = f.read()
    except FileNotFoundError:
        return pos
    if chunk:
        print(chunk.decode("utf-8", "replace"), end="", flush=True)
    return pos + len(chunk)


def run_stage(stage: str, cfg: Config, plot: int = 0, isolate: bool = True) -> None:
    """isolate=True: stage chay trong process con => JVM/torch tra RAM cho HDH khi stage xong.

    Khong dung subprocess + __file__ (notebook khong co __file__) va khong dung spawn
    (ham dinh nghia trong notebook khong pickle duoc) => dung fork (Linux/Colab).
    Luu y: kernel (process cha) khong duoc tu import torch/khoi dong Spark truoc do, neu khong
    isolate se khong con y nghia -> neu da lam vay hay Restart runtime.
    """
    if not isolate or sys.platform != "linux":
        _call_stage(stage, cfg, plot)
        return

    os.makedirs(cfg.out_dir, exist_ok=True)
    log = os.path.join(cfg.out_dir, f"log_{stage}.txt")
    if os.path.exists(log):
        os.remove(log)

    p = mp.get_context("fork").Process(target=_stage_child, args=(stage, cfg, plot, log))
    p.start()
    pos = 0
    try:
        while p.is_alive():
            p.join(timeout=1.0)
            pos = _drain(log, pos)         # in log theo thoi gian thuc
        _drain(log, pos)
    finally:
        if p.is_alive():                   # bam Stop trong notebook => dung luon process con
            p.terminate()
            p.join()
    if p.exitcode != 0:
        raise RuntimeError(f"Stage '{stage}' failed (exit code {p.exitcode}). Xem {log}")


def run(stage: str = "all", *, out_dir: str | None = None, n_docs: int | None = None,
        master: str | None = None, threshold: float | None = None, nlp_mem: str | None = None,
        no_embed: bool = False, plot: int = 0, isolate: bool = True,
        shared_spark: bool = False) -> Config:
    """Ham chinh de goi tu notebook, vd:  run("all", n_docs=1000, threshold=0.88)"""
    over = {"out_dir": out_dir, "n_docs": n_docs, "master": master, "sim_threshold": threshold,
            "nlp_driver_mem": nlp_mem, "use_mapping": False if no_embed else None}
    cfg = replace(Config(), **{k: v for k, v in over.items() if v is not None})

    stages = STAGES if stage == "all" else [stage]
    if no_embed:
        stages = [s for s in stages if s != "embed"]
    if shared_spark:
        # 1 SparkSession duy nhat cho moi stage (chay trong process hien tai, khong fork).
        # Session phai co du jar cho moi stage va heap = muc lon nhat. Doi lai JVM (+ model NLP) nam lai
        # RAM suot cac stage sau, ke ca luc embed chay torch => hop voi cluster, KHONG hop muc 2-3 GB tren 1 may.
        spark = None
        if any(s in ("nlp", "graph", "analyze") for s in stages):
            spark = build_spark(cfg, "kg", cfg.nlp_driver_mem, [SPARKNLP_PKG, GRAPHFRAMES_PKG])
        try:
            for s in stages:
                print(f"\n################ STAGE: {s} ################", flush=True)
                _call_stage(s, cfg, plot, spark)
        finally:
            if spark is not None:
                spark.stop()
        return cfg

    for s in stages:
        print(f"\n################ STAGE: {s} ################", flush=True)
        run_stage(s, cfg, plot, isolate)
    return cfg


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--stage", default="all", choices=["all"] + STAGES)
    ap.add_argument("--out-dir")
    ap.add_argument("--n-docs", type=int)
    ap.add_argument("--master")
    ap.add_argument("--threshold", type=float, help="similarity threshold cho entity normalization")
    ap.add_argument("--nlp-mem", help="driver heap cho stage nlp, vd 1536m")
    ap.add_argument("--no-embed", action="store_true", help="bo qua chuan hoa embedding (canh = ten viet thuong)")
    ap.add_argument("--plot", type=int, default=0, help="ve top-N nut theo degree trong stage analyze")
    ap.add_argument("--no-isolate", action="store_true", help="chay stage ngay trong process hien tai")
    ap.add_argument("--shared-spark", action="store_true", help="1 SparkSession dung chung cho cac stage Spark")
    args, _ = ap.parse_known_args(argv)    # bo qua tham so la (vd '-f kernel.json' cua Jupyter)

    run(args.stage, out_dir=args.out_dir, n_docs=args.n_docs, master=args.master,
        threshold=args.threshold, nlp_mem=args.nlp_mem, no_embed=args.no_embed,
        plot=args.plot, isolate=not args.no_isolate, shared_spark=args.shared_spark)


if __name__ == "__main__" and not _in_notebook():
    main()

In [3]:
# -*- coding: utf-8 -*-
"""
Knowledge Graph theo kieu MAP-REDUCE: moi "stage" in ra ro no la SPLIT / MAP / SHUFFLE+REDUCE / ...

Doi chieu thuat ngu Map-Reduce <-> Spark (de doc log):
  SPLIT           : chia input thanh cac manh. Spark: repartition(n) => 1 partition = 1 split = 1 map task
  MAP             : bien doi tung dong/tung manh, KHONG trao doi du lieu giua cac manh (narrow).
                    flatMap = 1 dong -> nhieu dong (explode)
  SHUFFLE         : day cac dong cung KEY ve cung 1 reducer. Trong plan Spark = "Exchange hashpartitioning"
  REDUCE          : gop cac dong cung key (groupBy(key).agg(...)). Spark tu COMBINE (gop cuc bo) truoc khi shuffle
  BROADCAST JOIN  : "map-side join": bang nho duoc copy toi moi worker, join ngay trong map task => KHONG shuffle
  DRIVER-SIDE     : buoc khong phai Map-Reduce, chay tren driver (vd embedding + gom cum)

Dung chung 1 SparkSession cho toan bo cac stage (build_spark chi goi 1 lan).
Chay: python kg_mapreduce_walkthrough.py   (hoac dan tung khoi `# %%` vao cac cell cua notebook)
"""
# %% 0. SETUP ------------------------------------------------------------------------------------
import contextlib
import os
import re
import sys
import time
from dataclasses import replace

import pandas as pd
from pyspark.sql import functions as F

try:                                   # script: them thu muc chua kg_pipeline.py; notebook: khong co __file__
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
except NameError:
    pass


def _flag(name: str) -> bool:
    return os.environ.get(f"KG_{name}", "1") == "1"


DO_SAMPLE = _flag("DO_SAMPLE")         # False => dung docs.parquet co san
DO_NER = _flag("DO_NER")               # False => dung occurrences co san (bo qua stage SPLIT + MAP NER)
DO_EMBED = _flag("DO_EMBED")           # False => dung entity_mapping co san, neu khong co thi canonical = ten viet thuong
DO_ANALYZE = _flag("DO_ANALYZE")       # PageRank / degree bang GraphFrames

cfg = replace(Config(), n_docs=100, sim_threshold=0.90)
os.makedirs(cfg.out_dir, exist_ok=True)

SUMMARY: list[dict] = []               # 1 dong / stage, in o cuoi


# %% helper: in "status" cua tung stage -------------------------------------------------------------
def plan_shape(df) -> str:
    """Dem so lan SHUFFLE va BROADCAST trong physical plan (plan ban dau cua AQE)."""
    plan = df._jdf.queryExecution().executedPlan().toString()
    n_shuffle = len(re.findall(r"Exchange (?:hashpartitioning|rangepartitioning|roundrobinpartitioning|SinglePartition)", plan))
    n_bcast = plan.count("BroadcastExchange")
    return f"plan: shuffle={n_shuffle} broadcast={n_bcast}"


def job_stats(spark, group: str):
    tr = spark.sparkContext.statusTracker()
    jobs = tr.getJobIdsForGroup(group)
    stages = {}
    for j in jobs:
        info = tr.getJobInfo(j)
        for sid in (info.stageIds if info else []):
            si = tr.getStageInfo(sid)
            if si:
                stages[sid] = si.numTasks
    return len(jobs), len(stages), sum(stages.values())


@contextlib.contextmanager
def phase(stage, kind: str, what: str, spark=None):
    """In dau/cuoi stage + so job/stage/task THAT ma Spark da chay trong stage do."""
    print(f"\n[STAGE {stage}] {kind}\n    {what}", flush=True)
    gid = f"kg-stage-{stage}"
    if spark is not None:
        spark.sparkContext.setJobGroup(gid, f"{kind}: {what}"[:100])
    t0 = time.time()
    yield
    sec = time.time() - t0
    jobs, stages, tasks = job_stats(spark, gid) if spark is not None else (0, 0, 0)
    print(f"    -> xong sau {sec:.1f}s | spark: jobs={jobs} stages={stages} tasks={tasks}", flush=True)
    SUMMARY.append({"stage": stage, "kind": kind, "seconds": round(sec, 1),
                    "jobs": jobs, "stages": stages, "tasks": tasks})


# %% STAGE 0: mot SparkSession duy nhat -----------------------------------------------------------
print("[STAGE 0] SPARK SESSION (tao 1 lan, dung chung cho moi stage)", flush=True)
spark = build_spark(cfg, "kg-mapreduce", cfg.nlp_driver_mem, [SPARKNLP_PKG, GRAPHFRAMES_PKG])
print(f"    master={cfg.master} | driver heap={cfg.nlp_driver_mem} | partitions={cfg.partitions}")

try:
    # %% STAGE 1: LOAD SAMPLE (driver-side, khong phai Map-Reduce) -------------------------------
    if DO_SAMPLE:
        with phase(1, "LOAD SAMPLE (driver-side)", f"doc {cfg.n_docs} bai Wikipedia -> docs.parquet"):
            stage_sample(cfg)

    if DO_NER:
        # %% STAGE 2: INPUT SPLIT ----------------------------------------------------------------
        with phase(2, "INPUT SPLIT", f"repartition({cfg.partitions}): moi split = 1 map task", spark):
            docs = (spark.read.parquet(P(cfg, "docs.parquet"))
                    .select("document_id", "content")
                    .repartition(cfg.partitions))
            sizes = (docs.groupBy(F.spark_partition_id().alias("split")).count()
                     .orderBy("split").collect())
            print("    docs / split:", {r["split"]: r["count"] for r in sizes})

        # %% STAGE 3: MAP 1 - NER (flatMap: 1 bai -> n thuc the) ---------------------------------
        with phase(3, "MAP 1 (flatMap) - NER",
                   "moi split: doc -> cau -> token -> GloVe -> NER -> emit (doc, cau, type, ten)", spark):
            model = ner_pipeline(cfg).fit(docs)                      # Transformer san co => khong chay job
            occ = occurrences_from(model.transform(docs), cfg)
            print("   ", plan_shape(occ))                              # shuffle=1 la repartition cua stage 2
            occ.write.mode("overwrite").parquet(P(cfg, "occurrences"))

    # %% STAGE 4: SHUFFLE + REDUCE 1 (word-count tren thuc the) ------------------------------------
    with phase(4, "SHUFFLE + REDUCE 1", "key=(entity_type, name_norm) -> value=count  => entity_freq", spark):
        occ = spark.read.parquet(P(cfg, "occurrences"))
        freq = entity_freq_from(occ)
        print("   ", plan_shape(freq))
        freq.write.mode("overwrite").parquet(P(cfg, "entity_freq"))
        print(f"    occurrences={occ.count()} | unique (type,name)={spark.read.parquet(P(cfg, 'entity_freq')).count()}")

    # %% STAGE 5: DRIVER-SIDE - chuan hoa thuc the bang embedding -----------------------------------
    if DO_EMBED:
        with phase(5, "DRIVER-SIDE (khong phai MR)",
                   "embedding + gom cum theo entity_type -> entity_mapping (se duoc broadcast o stage 6)"):
            stage_embed(cfg)
    map_path = P(cfg, "entity_mapping.parquet")
    use_mapping = os.path.exists(map_path)
    print(f"\n[STAGE 5] mapping: {'dung ' + map_path if use_mapping else 'khong co -> canonical = ten viet thuong'}")

    # %% STAGE 6: MAP 2 - BROADCAST (map-side) JOIN ---------------------------------------------------
    with phase(6, "MAP 2 - BROADCAST JOIN", "gan id = <TYPE>::<canonical>; bang mapping duoc copy toi moi map task", spark):
        mapping = spark.read.parquet(map_path) if use_mapping else None
        occ_id = with_entity_id(spark.read.parquet(P(cfg, "occurrences")), mapping)
        print("   ", plan_shape(occ_id), "(mong doi shuffle=0: join khong can shuffle)")
        occ_id.write.mode("overwrite").parquet(P(cfg, "occurrences_id"))
    occ_id = spark.read.parquet(P(cfg, "occurrences_id"))

    # %% STAGE 7: SHUFFLE + REDUCE 2 - vertices ---------------------------------------------------------
    with phase(7, "SHUFFLE + REDUCE 2", "2 reduce noi tiep: (id,type,surface)->count, roi (id,type)->sum + ten pho bien nhat", spark):
        vertices = vertices_from(occ_id)
        print("   ", plan_shape(vertices))
        vertices.write.mode("overwrite").parquet(P(cfg, "vertices"))

    # %% STAGE 8: SHUFFLE + REDUCE 3 - gom thuc the theo cau ----------------------------------------------
    with phase(8, "SHUFFLE + REDUCE 3", "key=(document_id, cau) -> value=mang id thuc the (da sort, bo cau > max)", spark):
        per_sentence = sentence_entities_from(occ_id, cfg)
        print("   ", plan_shape(per_sentence))
        per_sentence.write.mode("overwrite").parquet(P(cfg, "sentence_entities"))
        print(f"    cau co >= 2 thuc the: {spark.read.parquet(P(cfg, 'sentence_entities')).count()}")

    # %% STAGE 9: MAP 3 (flatMap) - sinh cap ----------------------------------------------------------------
    with phase(9, "MAP 3 (flatMap)", "moi cau k thuc the -> k(k-1)/2 cap (src<dst) => edge_evidence", spark):
        pairs = pairs_from(spark.read.parquet(P(cfg, "sentence_entities")))
        print("   ", plan_shape(pairs), "(mong doi shuffle=0: map-only job)")
        pairs.write.mode("overwrite").parquet(P(cfg, "edge_evidence"))

    # %% STAGE 10: SHUFFLE + REDUCE 4 - canh -------------------------------------------------------------------
    with phase(10, "SHUFFLE + REDUCE 4", "key=(src, dst) -> value=(so cau, so tai lieu) => edges", spark):
        edges = edges_from(spark.read.parquet(P(cfg, "edge_evidence")))
        print("   ", plan_shape(edges))
        edges.write.mode("overwrite").parquet(P(cfg, "edges"))

    # %% STAGE 11: VALIDATE (action = reduce ve driver) ------------------------------------------------------------
    with phase(11, "VALIDATE (reduce -> driver)", "dem + kiem tra canh treo (left_anti = shuffle join) + self-loop", spark):
        v = spark.read.parquet(P(cfg, "vertices"))
        e = spark.read.parquet(P(cfg, "edges"))
        n_v, n_e = v.count(), e.count()
        dangling = (e.select("src").union(e.select("dst")).distinct()
                    .join(v.select(F.col("id").alias("src")), "src", "left_anti").count())
        self_loops = e.where(F.col("src") == F.col("dst")).count()
        print(f"    Vertices={n_v} | Edges={n_e} | dangling={dangling} | self_loops={self_loops}")
        if dangling or self_loops:
            raise ValueError("Invalid Knowledge Graph")

    # %% STAGE 12: ANALYZE - lap nhieu vong (moi vong PageRank = join + reduceByKey) ---------------------------------
    if DO_ANALYZE:
        with phase(12, "ITERATIVE REDUCE (GraphFrames)", "degree + PageRank 10 vong + export Gephi", spark):
            stage_analyze(cfg, plot_top=0, spark=spark)

finally:
    # %% STAGE 13: giai phong -----------------------------------------------------------------------------------------
    print("\n[STAGE 13] STOP SPARK SESSION")
    spark.stop()

print("\n================ TONG KET THEO STAGE ================")
print(pd.DataFrame(SUMMARY).to_string(index=False))

[STAGE 0] SPARK SESSION (tao 1 lan, dung chung cho moi stage)
    master=local[2] | driver heap=1536m | partitions=8

[STAGE 1] LOAD SAMPLE (driver-side)
    doc 100 bai Wikipedia -> docs.parquet


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

docs: 100 | avg chars: 2345 | max chars: 28272
    -> xong sau 21.6s | spark: jobs=0 stages=0 tasks=0

[STAGE 2] INPUT SPLIT
    repartition(8): moi split = 1 map task
    docs / split: {0: 12, 1: 12, 2: 13, 3: 13, 4: 13, 5: 13, 6: 12, 7: 12}
    -> xong sau 10.8s | spark: jobs=6 stages=14 tasks=42

[STAGE 3] MAP 1 (flatMap) - NER
    moi split: doc -> cau -> token -> GloVe -> NER -> emit (doc, cau, type, ten)
glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[OK!]
ner_dl download started this may take some time.
Approximate size to download 13.6 MB
[OK!]
    plan: shuffle=0 broadcast=0
    -> xong sau 54.0s | spark: jobs=7 stages=8 tasks=37

[STAGE 4] SHUFFLE + REDUCE 1
    key=(entity_type, name_norm) -> value=count  => entity_freq
    plan: shuffle=1 broadcast=0
    occurrences=4552 | unique (type,name)=3152
    -> xong sau 1.1s | spark: jobs=8 stages=11 tasks=15

[STAGE 5] DRIVER-SIDE (khong phai MR)
    embedding + gom cum theo entity_type 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  LOC: 614 names -> 594 clusters
  MISC: 524 names -> 496 clusters
  ORG: 1004 names -> 939 clusters
  PER: 1010 names -> 981 clusters
merged names: 142 (xem merged_entities.csv)
entity_type  canonical                      
PER          m'millan                           4
ORG          mgtow                              2
LOC          wakayama                           2
ORG          biennale d'arte triveneta          2
             biennale nazionale d'arte sacra    2
MISC         usl league                         2
ORG          supreme court                      2
             mississippi delta chapter          2
MISC         fiba under-17                      2
PER          péter kőszeghy                     2
dtype: int64

===== RESOURCE [embed] =====
seconds         : 33.2
rss_start_mb    : 1998.6
rss_end_mb      : 2404.1
peak_rss_mb     : 2415.0
peak_python_mb  : 1011.1
peak_java_mb    : 1407.7
jvm_gc_ms       : 0
jvm_gc_count    : 0
    -> xong sau 33.2s | spark: jobs=0 stages=